In [2]:
import warnings
warnings.filterwarnings("ignore")

from rfdetr import RFDETRLarge
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import os
import optuna
import json

In [2]:
best_model = Path(
    f"runs/optuna/encoder/trial_1"
) 
with open(best_model / "training_config.json", "r") as f:
    best_params = json.load(f)["train_config"]

print(f"Best lr: {best_params['lr']}")
print(f"Best lr_encoder: {best_params['lr_encoder']}")
print(f"Best weight_decay: {best_params['weight_decay']}")

Best lr: 6.964583207205953e-05
Best lr_encoder: 0.00024250150081520832
Best weight_decay: 1.54603838402487e-05


In [3]:
def objective_aug(trial):

    # Fixed best hyperparameters from previous Optuna study
    lr = best_params["lr"]
    lr_encoder = best_params["lr_encoder"]
    weight_decay = best_params["weight_decay"]

    # Tune augmentation
    flip_p = trial.suggest_float("flip_p", 0.2, 0.7)
    rotate_limit = trial.suggest_int("rotate_limit", 0, 12)
    rotate_p = trial.suggest_float("rotate_p", 0.0, 0.4)

    brightness_p = trial.suggest_float(
        "brightness_p", 0.0, 0.5
    )

    blur_p = trial.suggest_float(
        "blur_p", 0.0, 0.2
    )

    aug_config = {
        "HorizontalFlip": {
            "p": flip_p
        },

        "Rotate": {
            "limit": rotate_limit,
            "p": rotate_p
        },

        "RandomBrightnessContrast": {
            "brightness_limit": 0.2,
            "contrast_limit": 0.2,
            "p": brightness_p
        },

        "GaussianBlur": {
            "p": blur_p
        },
    }

    model = RFDETRLarge(
        pretrain_weights=best_model / "checkpoint_best_total.pth",
        resolution=336,
        gradient_checkpointing=True
    )

    output_dir = (
        f"runs/optuna/augmentation/trial_{trial.number}"
    )

    model.train(
        dataset_dir=r"data/rfdetr",
        epochs=40,
        batch_size=4,
        grad_accum_steps=8,

        lr=lr,
        lr_encoder=lr_encoder,
        weight_decay=weight_decay,
        lr_scheduler="cosine",

        use_ema=True,

        early_stopping=True,
        early_stopping_patience=5,
        early_stopping_min_delta=1e-3,
        skip_best_epochs=5,

        eval_interval=1,
        seed=42,

        aug_config=aug_config,
        augmentation_backend="albumentations",

        output_dir=output_dir
    )

    metrics = model.evaluate(
        dataset_dir=r"data/rfdetr",
        split="val"
    )

    return metrics["val/mAP_50"]

In [4]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(sampler= sampler, direction="maximize")
study.optimize(objective_aug, n_trials=8)

Val (Epoch 1/100) — Overall Metrics               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.4989 │ 0.7174 │ 0.5414 │ 0.6984 │ 0.6991 │ 0.7914 │ 0.6318 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/F1           │    0.6991430521011353     │
│        val/mAP_50         │    0.7173739671707153     │
│       val/mAP_50_95       │    0.49887439608573914    │
│        val/mAP_75         │    0.5413585305213928     │
│          val/mAR          │    0.6984079480171204     │
│       val/precision       │    0.7913832664489746     │
│        val/recall         │    0.6317653656005859     │
└───────────────────────────┴───────────────────────────┘

[I 2026-09-29 23:00:08,786] Trial 7 finished with value: 0.7173739671707153 and parameters: {'flip_p': 0.6041986740582306, 'rotate_limit': 3, 'rotate_p': 0.039068845602553554, 'brightness_p': 0.34211651325607845, 'blur_p': 0.08803049874792027}. Best is trial 4 with value: 0.7206959128379822.


In [5]:
best_trial = study.best_trial
print(best_trial.number)
print(best_trial.value)
print(best_trial.params)

4
0.7206959128379822
{'flip_p': 0.5059264473611897, 'rotate_limit': 1, 'rotate_p': 0.11685785941408727, 'brightness_p': 0.18318092164684585, 'blur_p': 0.0912139968434072}


In [3]:
# 1(best) 4 5 7 8 high
# 2 3 6 low
NUM_CLASSES = 8

# best_checkpoint = Path(
#     f"runs/optuna/augmentation/trial_{best_trial.number}"
# ) 
best_checkpoint = Path(
    f"runs/optuna/augmentation/trial_7"
) 

model = RFDETRLarge(
    pretrain_weights= best_checkpoint / "checkpoint_best_total.pth"
)

image_dir = Path(r"data/raw/test/test")

rows = []
row_id = 0

for image_path in tqdm(list(image_dir.glob("*")), desc="Predicting"):

    detections = model.predict(
        str(image_path),
        threshold=0.01
    )

    for box, conf, cls_id in zip(
        detections.xyxy,
        detections.confidence,
        detections.class_id
    ):
        cls_id = int(cls_id)

        # Skip background / invalid class
        if not 0 <= cls_id < NUM_CLASSES:
            continue

        x1, y1, x2, y2 = map(float, box)

        # Skip degenerate boxes
        if x2 <= x1 or y2 <= y1:
            continue

        rows.append({
            "id": row_id,
            "image_id": image_path.name,
            "class_id": cls_id,
            "confidence": float(conf),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

        row_id += 1

df_pred = pd.DataFrame(
    rows,
    columns=[
        "id",
        "image_id",
        "class_id",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
    ]
)

# Final validation
assert df_pred["class_id"].between(0, 7).all()
assert (df_pred["x2"] > df_pred["x1"]).all()
assert (df_pred["y2"] > df_pred["y1"]).all()

df_pred.to_csv(
    best_checkpoint / "submission.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Predictions:", len(df_pred))

[2026-09-30 15:50:04] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-30 15:50:04] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-30 15:50:04] [WARNING] rf-detr - Checkpoint has 8 classes but model is configured for 90. Using checkpoint class count (8). Pass num_classes=8 to suppress this warning.
Predicting: 100%|██████████| 1013/1013 [00:42<00:00, 23.82it/s]


Predictions: 278692
